In [2]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=512
    )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
train_size = 20000
test_size = 5000

train_dataset = ds["train"].select(range(min(train_size, len(ds["train"]))))
test_dataset = ds["test"].select(range(min(test_size, len(ds["test"]))))

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [7]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {
        'accuracy': accuracy,
        'f1': f1
    }

training_args = TrainingArguments(
    output_dir="./results",          # куда сохранять результаты
    num_train_epochs=3,              # количество эпох
    per_device_train_batch_size=16,  # размер батча для обучения
    per_device_eval_batch_size=64,   # размер батча для оценки
    warmup_steps=500,                # шаги разогрева
    weight_decay=0.01,
    logging_steps=100,               # логировать каждые 100 шагов
    eval_strategy="epoch",     # оценивать после каждой эпохи
    save_strategy="epoch",           # сохранять модель после каждой эпохи
    load_best_model_at_end=True,     # загрузить лучшую модель в конце
    metric_for_best_model="accuracy", # метрика для выбора лучшей модели
    fp16=True if torch.cuda.is_available() else False,  # ускорение на GPU
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.466531,0.376215,0.916400,0.000000
2,0.279484,0.528862,0.905000,0.000000
3,0.100889,0.415315,0.950000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1875, training_loss=0.36481126759847005, metrics={'train_runtime': 1662.8549, 'train_samples_per_second': 36.083, 'train_steps_per_second': 1.128, 'total_flos': 7948043919360000.0, 'train_loss': 0.36481126759847005, 'epoch': 3.0})

In [9]:
results = trainer.evaluate()
print(f"\nРезультаты на тестовой выборке:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1-score: {results['eval_f1']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Результаты на тестовой выборке:
Accuracy: 0.9500
F1-score: 0.0000


In [11]:
def predict_sentiment(text, model, tokenizer):
    """Предсказывает тональность текста"""
    model.eval()
    
    # Токенизируем (тензоры остаются на CPU)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
    
    # Определяем устройство модели и переносим данные на него
    device = next(model.parameters()).device  # получаем устройство модели (cuda или cpu)
    
    # Переносим все входные тензоры на то же устройство
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=1).item()
        confidence = predictions[0][predicted_class].item()
    
    sentiment = "Позитивный" if predicted_class == 1 else "Негативный"
    return sentiment, confidence

# Примеры предсказаний
print("\n" + "="*50)
print("Примеры предсказаний:")
print("="*50)

test_reviews = [
    "This movie was absolutely fantastic! Great acting and amazing plot.",
    "Terrible film, waste of time. I regret watching it.",
    "It was okay, nothing special but not terrible either.",
    "Best movie I've seen this year! Highly recommend!",
    "Poor directing and boring story. Disappointing."
]

for review in test_reviews:
    sentiment, confidence = predict_sentiment(review, model, tokenizer)
    print(f"\nОтзыв: {review[:80]}...")
    print(f"Тональность: {sentiment} (уверенность: {confidence:.2%})")


Примеры предсказаний:

Отзыв: This movie was absolutely fantastic! Great acting and amazing plot....
Тональность: Позитивный (уверенность: 99.80%)

Отзыв: Terrible film, waste of time. I regret watching it....
Тональность: Негативный (уверенность: 99.91%)

Отзыв: It was okay, nothing special but not terrible either....
Тональность: Негативный (уверенность: 98.66%)

Отзыв: Best movie I've seen this year! Highly recommend!...
Тональность: Позитивный (уверенность: 99.79%)

Отзыв: Poor directing and boring story. Disappointing....
Тональность: Негативный (уверенность: 99.89%)
